In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
%matplotlib inline

In [ ]:
df = pd.read_csv('data/RRCA_baseflow.csv')

# Convert date (subtract 693963 for days since Jan 1 1900)
df['Days'] = df['Date'] - 693963
df['DateTime'] = pd.to_datetime('1900-01-01') + pd.to_timedelta(df['Days'], unit='D')
df['Year'] = df['DateTime'].dt.year

# Pumping is negative in the data (water leaving aquifer), flip so positive = more pumping
df['Pumping'] = -df['Irrigation_pumping']

print(f'Rows: {len(df)}, Segments: {df["Segment_id"].nunique()}')
print(f'Date range: {df["DateTime"].min().date()} to {df["DateTime"].max().date()}')
print(f'Missing values: {df.isna().sum().sum()}')
df.head()

## Exploratory Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].scatter(df['Pumping'], df['Observed'], alpha=0.15, s=10)
axes[0].set_xlabel('Pumping')
axes[0].set_ylabel('Observed Baseflow')
axes[0].set_title('Pumping vs Baseflow')

axes[1].scatter(df['Precipitation'], df['Observed'], alpha=0.15, s=10, color='green')
axes[1].set_xlabel('Precipitation')
axes[1].set_ylabel('Observed Baseflow')
axes[1].set_title('Precipitation vs Baseflow')

axes[2].scatter(df['Evapotranspiration'], df['Observed'], alpha=0.15, s=10, color='orange')
axes[2].set_xlabel('Evapotranspiration')
axes[2].set_ylabel('Observed Baseflow')
axes[2].set_title('ET vs Baseflow')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

pairs = [('Evapotranspiration', 'Precipitation'), ('Evapotranspiration', 'Irrigation_pumping'),
         ('Evapotranspiration', 'Observed'), ('Precipitation', 'Irrigation_pumping'),
         ('Precipitation', 'Observed'), ('Irrigation_pumping', 'Observed')]

for ax, (x_col, y_col) in zip(axes.flat, pairs):
    ax.scatter(df[x_col], df[y_col], alpha=0.15, s=10)
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)

plt.tight_layout()
plt.show()

In [ ]:
corr = df[['Pumping', 'Precipitation', 'Evapotranspiration', 'Observed']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
attributes = ['Evapotranspiration', 'Precipitation', 'Irrigation_pumping', 'Observed']

plt.figure(figsize=(12, 5))
for attr in attributes:
    yearly_avg = df.groupby('Year')[attr].mean()
    normalized = (yearly_avg - yearly_avg.mean()) / yearly_avg.std()
    plt.plot(yearly_avg.index, normalized.values, label=attr)

plt.xlabel('Year')
plt.ylabel('Standardized Value (z-score)')
plt.title('All Attributes Over Time (Normalized)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
seg_stats = df.groupby('Segment_id').agg(x=('x', 'first'), y=('y', 'first'),
                                          mean_baseflow=('Observed', 'mean')).reset_index()

plt.figure(figsize=(8, 6))
sc = plt.scatter(seg_stats['x'], seg_stats['y'], c=seg_stats['mean_baseflow'],
                 cmap='YlGnBu', s=120, edgecolors='black', linewidth=0.5)
plt.colorbar(sc, label='Mean Baseflow')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Segment Locations Colored by Mean Baseflow')
plt.tight_layout()
plt.show()

In [ ]:
seg_means = df.groupby('Segment_id')['Observed'].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

seg_means.plot(kind='bar', ax=axes[0], edgecolor='black')
axes[0].set_ylabel('Mean Baseflow')
axes[0].set_title('Mean Baseflow by Segment')
axes[0].tick_params(axis='x', labelsize=7, rotation=90)

# Compare high vs low baseflow segments
top3 = seg_means.head(3).index.tolist()
bot3 = seg_means.tail(3).index.tolist()
for seg in top3 + bot3:
    sub = df[df['Segment_id'] == seg]
    axes[1].scatter(sub['Precipitation'], sub['Observed'], s=8, alpha=0.4, label=f'Seg {seg}')

axes[1].set_xlabel('Precipitation')
axes[1].set_ylabel('Observed Baseflow')
axes[1].set_title('High vs Low Baseflow Segments')
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.show()

## Multiple Linear Regression (OLS)

Model: Observed = β₀ + β₁·Pumping + β₂·Precip + β₃·ET + β₄·(Pumping×Precip) + β₅·(Pumping×ET) + segment dummies

Interaction terms let the pumping effect change with climate. Segment dummies control for different baseline flow levels across the 42 river segments — without them R² is only ~0.09.

In [ ]:
X = df[['Pumping', 'Precipitation', 'Evapotranspiration']].copy()
X['Pump_x_Precip'] = X['Pumping'] * X['Precipitation']
X['Pump_x_ET'] = X['Pumping'] * X['Evapotranspiration']

segment_dummies = pd.get_dummies(df['Segment_id'], prefix='seg', drop_first=True, dtype=float)
X = pd.concat([X, segment_dummies], axis=1)
X = sm.add_constant(X)
y = df['Observed']

ols_model = sm.OLS(y, X).fit()

key_vars = ['const', 'Pumping', 'Precipitation', 'Evapotranspiration', 'Pump_x_Precip', 'Pump_x_ET']
print(f'R-squared: {ols_model.rsquared:.4f}')
print(f'Observations: {int(ols_model.nobs)}\n')
for v in key_vars:
    print(f'  {v:25s}  coeff={ols_model.params[v]:8.4f}  p={ols_model.pvalues[v]:.4f}')

In [ ]:
sns.regplot(x='Pumping', y='Observed', data=df[df['Pumping'] > 0],
            scatter_kws={'alpha': 0.2, 's': 10}, line_kws={'color': 'red'})
plt.ylim(-10, 150)
plt.title('Pumping vs Baseflow with Regression Line')
plt.tight_layout()
plt.show()

## Train/Test Validation (sklearn)

Same features and segment dummies, but using sklearn's LinearRegression with an 80/20 train/test split to check how well the model generalizes.

In [ ]:
features = ['Pumping', 'Precipitation', 'Evapotranspiration']
seg_dummies = pd.get_dummies(df['Segment_id'], prefix='Seg', drop_first=True, dtype=float)

X_sk = pd.concat([df[features], seg_dummies], axis=1)
y_sk = df['Observed']

X_train, X_test, y_train, y_test = train_test_split(X_sk, y_sk, test_size=0.2, random_state=42)

reg = LinearRegression()
reg.fit(X_train, y_train)

y_pred = reg.predict(X_test)
print(f'Test R²: {r2_score(y_test, y_pred):.4f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}')

In [ ]:
labels = ['Pumping', 'Precipitation', 'Evapotranspiration']
coefs = reg.coef_[:3]
colors = ['red' if c < 0 else 'green' for c in coefs]

plt.figure(figsize=(7, 4))
plt.barh(labels, coefs, color=colors, edgecolor='black')
plt.axvline(0, color='black', linewidth=1)
plt.xlabel('Effect on Baseflow (m³/day per unit change)')
plt.title('Regression Coefficients')
plt.tight_layout()
plt.show()

## Model Diagnostics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].scatter(ols_model.fittedvalues, ols_model.resid, alpha=0.1, s=5)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Fitted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted')

axes[1].hist(ols_model.resid, bins=60, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution')

axes[2].scatter(y, ols_model.fittedvalues, alpha=0.1, s=5)
axes[2].plot([0, y.max()], [0, y.max()], 'r--')
axes[2].set_xlabel('Actual')
axes[2].set_ylabel('Predicted')
axes[2].set_title('Actual vs Predicted')

plt.tight_layout()
plt.show()

## Predicting Safe Irrigation Pumping

Since the model predicts baseflow from pumping and climate, we can flip it around: how much pumping can happen before baseflow drops to zero?

$$P_{threshold} = -\frac{\beta_0 + \beta_2 \cdot Precip + \beta_3 \cdot ET}{\beta_1 + \beta_4 \cdot Precip + \beta_5 \cdot ET}$$

In [ ]:
b0 = ols_model.params['const']
b1 = ols_model.params['Pumping']
b2 = ols_model.params['Precipitation']
b3 = ols_model.params['Evapotranspiration']
b4 = ols_model.params['Pump_x_Precip']
b5 = ols_model.params['Pump_x_ET']

def pumping_threshold(precip, et):
    denom = b1 + b4 * precip + b5 * et
    if denom == 0:
        return np.inf
    return max(-(b0 + b2 * precip + b3 * et) / denom, 0)

precip_q = df['Precipitation'].quantile([0.25, 0.5, 0.75])
et_q = df['Evapotranspiration'].quantile([0.25, 0.5, 0.75])

rows = []
for p_name, p_val in zip(['Low', 'Avg', 'High'], precip_q):
    for e_name, e_val in zip(['Low', 'Avg', 'High'], et_q):
        rows.append({'Precip': p_name, 'ET': e_name,
                     'Max Safe Pumping': round(pumping_threshold(p_val, e_val), 4)})
pd.DataFrame(rows)

In [ ]:
precip_range = np.linspace(df['Precipitation'].quantile(0.10), df['Precipitation'].quantile(0.90), 50)
et_range = np.linspace(df['Evapotranspiration'].quantile(0.10), df['Evapotranspiration'].quantile(0.90), 50)

grid = np.zeros((len(et_range), len(precip_range)))
for i, et in enumerate(et_range):
    for j, precip in enumerate(precip_range):
        denom = b1 + b4 * precip + b5 * et
        grid[i, j] = pumping_threshold(precip, et) if denom < 0 else np.nan

plt.figure(figsize=(10, 7))
im = plt.imshow(grid, origin='lower', aspect='auto',
                extent=[precip_range[0], precip_range[-1], et_range[0], et_range[-1]],
                cmap='RdYlGn', vmin=0, vmax=2)
plt.colorbar(im, label='Max Safe Pumping')
plt.xlabel('Precipitation')
plt.ylabel('Evapotranspiration')
plt.title('Predicted Safe Irrigation Pumping by Climate Conditions')
plt.tight_layout()
plt.show()

In [ ]:
safe_baseflow = y.quantile(0.10)

b0 = ols_model.params['const']
b_pump = ols_model.params['Pumping']
b_precip = ols_model.params['Precipitation']
b_et = ols_model.params['Evapotranspiration']

def max_safe_pumping(precip, et, safe_level=safe_baseflow):
    flow_without_pumping = b0 + b_precip * precip + b_et * et
    limit = (safe_level - flow_without_pumping) / b_pump
    return max(0, limit)

p_range = np.linspace(0, df['Precipitation'].quantile(0.95), 100)
mean_et = df['Evapotranspiration'].mean()
high_et = df['Evapotranspiration'].quantile(0.90)

plt.figure(figsize=(9, 5))
plt.plot(p_range, [max_safe_pumping(p, mean_et * 0.7) for p in p_range],
         label='Cool / Low ET (0.7x Mean)', color='#1f77b4', lw=2.2)
plt.plot(p_range, [max_safe_pumping(p, mean_et) for p in p_range],
         label='Average ET (Mean)', color='#2ca02c', lw=2.2)
plt.plot(p_range, [max_safe_pumping(p, high_et) for p in p_range],
         label='Hot / Drought ET (90th percentile)', color='#d62728', lw=2.2, linestyle='--')
plt.axhline(0, color='black', lw=1)
plt.title('Dynamic Safe Pumping Threshold Response to Precipitation & ET', fontsize=12, weight='bold')
plt.xlabel('Precipitation Amount (mm or Index)', fontsize=11)
plt.ylabel('Max Allowable Safe Pumping Rate', fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(title='Climate Conditions', loc='upper left')
plt.tight_layout()
plt.show()

## Conclusion

1. The model predicts baseflow with R² ≈ 0.81 using pumping, precipitation, ET, interaction terms, and segment fixed effects. The sklearn train/test split confirms it generalizes well.

2. Precipitation increases baseflow, ET decreases it, and pumping reduces it — all in the expected directions.

3. By inverting the model we can estimate safe irrigation pumping limits that adjust with climate — more pumping is sustainable during wet/cool periods, less during dry/hot periods.

4. Limitations: linear model, thresholds are for the baseline segment, and no time-lagged effects are captured.